In [ ]:
%load_ext autoreload
%autoreload 2

import torch
from typing import List, Dict
from datasets import load_dataset, Dataset
from sentiments_utils import LLMSentimentAnalyzer
import sentiments_utils as utils

In [ ]:
# Dataset configuration
DATASET_PATH = "parallel_datasets"
DATASET_SPLITS = {"train": f"{DATASET_PATH}/train.jsonl",
                  "val": f"{DATASET_PATH}/val.jsonl",
                  "test": f"{DATASET_PATH}/test.jsonl"}
# Language configuration
SOURCE_COLUMN = "shp" 
TARGET_COLUMN = "spa"
NUMBER_LABELS = 3  # Positive, Negative, Neutral
# General configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Leer datasets de entrenamiento, validación y prueba
full_datasets = utils.load_parallel_dataset_jsonl(DATASET_SPLITS["train"], DATASET_SPLITS["val"], DATASET_SPLITS["test"])
train_dataset = full_datasets["train"]
val_dataset = full_datasets["validation"]
test_dataset = full_datasets["test"]

In [ ]:
API_URL = "http://10.5.0.2:1234/v1"  # LM Studio default local server
API_KEY = "not-needed"  # LM Studio doesn't require an API key
MODEL = "openai/gpt-oss-20b"  # LM Studio uses whatever model is loaded

In [ ]:
# Initialize analyzer
analyzer = LLMSentimentAnalyzer(api_url=API_URL, api_key=API_KEY, model=MODEL, max_retries=3)

### Enviar a LLM para analisis

In [ ]:
 # Analyze dataset
print(f"Analyzing {len(train_dataset)} texts...\n")
analyzed_dataset = analyzer.analyze_dataset(train_dataset, TARGET_COLUMN, batch_size=20)

In [ ]:
# Save analyzed dataset to a JSONL file
analyzed_dataset.to_json("sentiment_datasets/train.jsonl")
analyzed_dataset.to_parquet("sentiment_datasets/train.parquet")

In [ ]:
dataset  = analyzed_dataset
text_column = TARGET_COLUMN
# Print resuts summary
print("\n" + "="*80)
print("SENTIMENT ANALYSIS RESULTS (First 5)")
print("="*80 + "\n")

for i in range(min(5, len(dataset))):
    example = dataset[i]
    print(f"{i+1}. Text: {example[text_column][:80]}...")
    print(f"   Sentiment: {example['sentiment'].upper()} (Confidence: {example['sentiment_confidence']})")
    print(f"   Explanation: {example['sentiment_explanation']}")
    print()        


# Show overall distribution of sentiments
sentiment_counts = {"positive": 0, "negative": 0, "neutral": 0}
for example in dataset:
    sentiment = example["sentiment"]
    if sentiment in sentiment_counts:
        sentiment_counts[sentiment] += 1
print("\n" + "="*80)
print("OVERALL SENTIMENT DISTRIBUTION")
print("="*80 + "\n")
for sentiment, count in sentiment_counts.items():
    print(f"{sentiment}: {count} ({(count/len(dataset))*100:.2f}%)")        

# Show random sample of positive, negative, and neutral texts with explanations
print("\n" + "="*80)
print("SAMPLE TEXTS BY SENTIMENT")
print("="*80 + "\n")
samples_shown = {"positive": 0, "negative": 0, "neutral": 0}
for example in dataset.shuffle(seed=42):
    sentiment = example['sentiment']
    if samples_shown[sentiment] < 5:
        print(f"Text: {example[text_column][:100]}...")
        print(f"Sentiment: {sentiment.upper()} (Confidence: {example['sentiment_confidence']})")
        print(f"Explanation: {example['sentiment_explanation']}\n")
        samples_shown[sentiment] += 1
    if all(count >= 5 for count in samples_shown.values()):
        break        


In [ ]:
analyzed_dataset